# Debug Tool Execution Issues — Fixes Verified

**Session**: `session_2026-03-22_20-17-42`

**Problems identified and fixed**:
1. `list_directory` with `recursive=true` returned overwhelming output (entire `.venv`) → **fixed** with `_EXCLUDE_DIRS`, `max_results`, `max_depth`
2. Agent tried `repo_browser.search` (hallucinated) and `search_files` with content patterns → **fixed** with new `grep_files` tool
3. Agent constructed wrong paths because it couldn't search file contents → **resolved** by `grep_files`

**Goal**: Run each cell to prove the fixes work.

In [5]:
# Setup environment
import sys
import os
import inspect
from pathlib import Path

project_root = Path("/Projects/agentX")
sys.path.insert(0, str(project_root / "src"))

os.environ["AGENTIX_HOME"] = str(project_root)

print(f"✓ Project root: {project_root}")
print(f"✓ AGENTIX_HOME: {os.environ['AGENTIX_HOME']}")

✓ Project root: /Projects/agentX
✓ AGENTIX_HOME: /Projects/agentX


## Fix 1: `list_directory` — Bounded, Filtered Recursive Traversal

**Before**: used `path.rglob()` with zero filtering — returned 50 000+ files including the entire `.venv` on project root.

**After**: skips `_EXCLUDE_DIRS` during traversal; enforces `max_results=500` and `max_depth=3`.

In [6]:
# Verify the fix is in place by inspecting the implementation
from agentx.integration.client_tool_executor import ClientToolExecutor, _EXCLUDE_DIRS

src = inspect.getsource(ClientToolExecutor._list_directory)

checks = {
    "Excludes noise dirs via _EXCLUDE_DIRS": "_EXCLUDE_DIRS" in src,
    "max_results parameter":                 "max_results"   in src,
    "max_depth parameter":                   "max_depth"     in src,
    "Truncation notice on overflow":         "truncated"     in src,
}

print("_list_directory implementation checks:")
for label, ok in checks.items():
    print(f"  {'✓' if ok else '❌'} {label}")

print(f"\nExcluded directories ({len(_EXCLUDE_DIRS)}):")
for d in sorted(_EXCLUDE_DIRS):
    print(f"  • {d}")

ImportError: cannot import name 'TOOL_REGISTRY' from 'shared.models.tools' (/Projects/agentX/src/shared/models/tools.py)

In [ ]:
# Reproduce the exact call that caused the production failure
from agentx.integration.client_tool_executor import list_directory

result    = list_directory("/Projects/agentX", recursive=True)
lines     = result.splitlines()
num_files = sum(1 for l in lines if l.startswith('[FILE]'))
num_dirs  = sum(1 for l in lines if l.startswith('[DIR]'))
has_venv  = any('.venv' in l for l in lines)
truncated = 'truncated' in result.lower()

print("list_directory('/Projects/agentX', recursive=True)")
print("=" * 60)
print(f"  Total lines   : {len(lines):,}")
print(f"  Files         : {num_files:,}")
print(f"  Directories   : {num_dirs:,}")
print(f"  Contains .venv: {has_venv}")
print(f"  Truncated     : {truncated}")
print()
if not has_venv and len(lines) < 1000:
    print('✓ PASS: Output is bounded and .venv is excluded')
else:
    print('❌ FAIL: Output still problematic')
print('\nFirst 20 lines:')
for line in lines[:20]:
    print(' ', line)

## Fix 2: `grep_files` — Content Search Tool

**Before**: `search_files` matched only *file names* by glob. Patterns like `parse_tool_name` and `AgentXSession` are identifiers inside files — every call returned zero results.

**After**: new `grep_files` tool searches *inside* files (like `grep -rn`), returning `file:line: content` matches.

In [ ]:
# Confirm grep_files is registered alongside the other client tools
from agentx.integration.client_tool_executor import CLIENT_TOOL_FUNCTIONS, get_client_tool_schemas

print('Registered client tools:')
for name in sorted(CLIENT_TOOL_FUNCTIONS.keys()):
    print(f'  ✓ {name}')

schemas      = get_client_tool_schemas()
schema_names = {s['function']['name'] for s in schemas}
grep_schema  = next((s for s in schemas if s['function']['name'] == 'grep_files'), None)

print(f'\nTool schemas ({len(schemas)} total): {sorted(schema_names)}')

if grep_schema:
    params = list(grep_schema['function']['parameters']['properties'].keys())
    print(f'\ngrep_files parameters: {params}')
    print('✓ PASS: grep_files registered with full OpenAI schema')
else:
    print('❌ FAIL: grep_files schema missing')

## Fix 3: grep_files Finds What search_files Could Not

The session log shows searches that returned empty results:
- `search_files` pattern `parse_tool_name` → `No files matching pattern`
- `search_files` pattern `AgentXSession` → `No files matching pattern`

These are identifiers inside files, not filenames. `grep_files` handles this correctly.

In [ ]:
# Reproduce the exact failed searches from the production session
from agentx.integration.client_tool_executor import search_files, grep_files

# Note: 'parse_tool_name' never existed — the session was requesting its creation.
# 'AgentXSession' is the class the agent was trying to locate.
SEARCHES = [
    ('AgentXSession', 'class the agent was looking for'),
    ('_list_directory', 'method the agent was trying to modify'),
]

for pattern, label in SEARCHES:
    print('=' * 60)
    print(f"Pattern: '{pattern}'  ({label})")

    old = search_files('/Projects/agentX/src', pattern, recursive=True)
    print(f'\n  search_files (filename glob)  → {old[:80]}')

    new  = grep_files('/Projects/agentX/src', pattern, file_pattern='*.py', limit=5)
    hits = [l for l in new.splitlines() if ':' in l and not l.startswith('No matches')]
    if hits:
        print(f'\n  grep_files   (content search) → {len(hits)} match(es)')
        for h in hits[:3]:
            print(f'    {h}')
        print('  ✓ PASS')
    else:
        print(f'\n  grep_files → {new[:80]}')
        print('  ❌ FAIL')
    print()

## End-to-End Validation

In [ ]:
import time
from agentx.integration.client_tool_executor import (
    list_directory, search_files, grep_files, _EXCLUDE_DIRS, CLIENT_TOOL_FUNCTIONS
)

results = {}

# Fix 1: list_directory bounded
r = list_directory('/Projects/agentX', recursive=True)
lines = r.splitlines()
results['list_directory bounded (<1000 lines)'] = len(lines) < 1000
results['list_directory excludes .venv']        = not any('.venv' in l for l in lines)

# Fix 1b: new parameters in signature
src = inspect.getsource(list_directory)
results['list_directory has max_results param'] = 'max_results' in src
results['list_directory has max_depth param']   = 'max_depth'   in src

# Fix 2: grep_files registered
results['grep_files in CLIENT_TOOL_FUNCTIONS']  = 'grep_files' in CLIENT_TOOL_FUNCTIONS

# Fix 3: grep_files finds content (the patterns the agent needed)
results["grep_files finds 'AgentXSession'"]     = not grep_files('/Projects/agentX/src', 'AgentXSession', file_pattern='*.py').startswith('No matches')
results["grep_files finds '_EXCLUDE_DIRS'"]     = not grep_files('/Projects/agentX/src', '_EXCLUDE_DIRS',   file_pattern='*.py').startswith('No matches')

# search_files no longer double-scans on limit overflow
t0 = time.time()
search_files('/Projects/agentX', '*.py', recursive=True, limit=50)
elapsed = time.time() - t0
results[f'search_files fast (<5s, took {elapsed:.2f}s)'] = elapsed < 5.0

print('=' * 60)
print('VALIDATION SUMMARY')
print('=' * 60)
all_pass = all(results.values())
for label, passed in results.items():
    print(f"  {'✓' if passed else '❌'} {label}")
print()
print('  ' + ('ALL CHECKS PASSED ✓' if all_pass else 'SOME CHECKS FAILED ❌'))

In [ ]:
# grep_files — regex and case-insensitive modes
print('grep_files — advanced options')
print('=' * 60)

r = grep_files('/Projects/agentX/src', r'class \w+Session', file_pattern='*.py', limit=5)
print('Regex \'class \\w+Session\':')
for line in r.splitlines()[:4]:
    print(f'  {line}')

print()
r = grep_files('/Projects/agentX/src', 'agentxsession', file_pattern='*.py', ignore_case=True, limit=3)
print("Case-insensitive 'agentxsession':")
for line in r.splitlines()[:3]:
    print(f'  {line}')

## Summary of Changes

| Issue from session log | Root Cause | Fix |
|---|---|---|
| `list_directory recursive=True` → 50k+ files | `rglob()` with zero filtering | `_EXCLUDE_DIRS`, `max_results=500`, `max_depth=3` |
| `search_files('parse_tool_name')` → no results | Tool matches filenames, not content | New `grep_files` tool searches file contents |
| Agent built wrong paths, couldn't find files | No content search available | `grep_files` locates definitions directly |

**New tool**: `grep_files(path, pattern, file_pattern='*.py', recursive=True, ignore_case=False, limit=200)`  
Registered in `CLIENT_TOOL_FUNCTIONS` and auto-exposed via `get_client_tool_schemas()`.

## Excluded Directories

The following names are skipped during all recursive traversal (`list_directory`, `search_files`, `grep_files`):

In [ ]:
from agentx.integration.client_tool_executor import _EXCLUDE_DIRS

print(f'_EXCLUDE_DIRS ({len(_EXCLUDE_DIRS)} entries):')
for name in sorted(_EXCLUDE_DIRS):
    print(f'  {name}')